In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Tabpfn

In [ ]:
# Install sklearn
!pip install scikit-learn==1.5.2

# Install TabPFN
!pip install tabpfn
!pip install tabpfn-client
# TabPFN Community installs optional functionalities around the TabPFN model
# These include post-hoc ensembles, interpretability tools, and more
!git clone https://github.com/PriorLabs/tabpfn-extensions
!pip install -e tabpfn-extensions[all]

In [ ]:
import os
import pandas as pd
from tabpfn import TabPFNRegressor
from sklearn.metrics import mean_squared_error, r2_score

def prepare_features(df, target_col, is_train=True, encoder_dict=None):
    """Automatically encode categorical features safely."""
    df_copy = df.copy()
    X = df_copy.drop(columns=[target_col])
    y = df_copy[target_col]

    if encoder_dict is None:
        encoder_dict = {}

    categorical_cols = X.select_dtypes(include=['object', 'category', 'bool']).columns

    for col in categorical_cols:
        if is_train:
            labels, uniques = pd.factorize(X[col])
            X[col] = labels
            encoder_dict[col] = uniques
        else:
            if col in encoder_dict:
                uniques = encoder_dict[col]
                mapping = {val: i for i, val in enumerate(uniques)}
                X[col] = X[col].map(mapping).fillna(-1).astype(int)
            else:
                X[col] = pd.factorize(X[col])[0]

    return X, y, encoder_dict

def main():
    # Setup data directories
    data_dir = "<path of data>"
    train_path = os.path.join(data_dir, "<path of training set>")
    valid_path = os.path.join(data_dir, "<path of valid set>")
    test_path = os.path.join(data_dir, "<path of test set>")

    print("Reading v3 datasets...")
    train_df = pd.read_csv(train_path)
    valid_df = pd.read_csv(valid_path)
    test_df = pd.read_csv(test_path)

    # Precisely lock target column alignment
    target_col = "<label>"
    print(f"Target column successfully locked: {target_col}")

    # Feature engineering for categorical data
    print("Processing feature profiles and encoding categorical columns...")
    X_train, y_train, encoders = prepare_features(train_df, target_col, is_train=True)
    X_valid, y_valid, _ = prepare_features(valid_df, target_col, is_train=False, encoder_dict=encoders)
    X_test, y_test, _ = prepare_features(test_df, target_col, is_train=False, encoder_dict=encoders)

    # Initialize TabPFN-3 Regressor
    model_v3_name = "tabpfn-v3-regressor-v3_default.ckpt"
    print(f"Loading latest TabPFN-3 regressor configuration: {model_v3_name}...")

    classifier = TabPFNRegressor(
        device='auto',
        model_path=model_v3_name
    )

    # Model fitting
    print("Fitting model via TabPFN-3 mixed-feature in-context learning...")
    classifier.fit(X_train, y_train)

    # Validation evaluation
    print("Generating validation set inferences...")
    valid_preds = classifier.predict(X_valid)
    print(f"Validation Mean Squared Error (MSE): {mean_squared_error(y_valid, valid_preds):.4f}")
    print(f"Validation R2 Coefficient of Determination: {r2_score(y_valid, valid_preds):.4f}")

    # Test evaluation
    print("Generating test set inferences...")
    test_preds = classifier.predict(X_test)
    test_mse = mean_squared_error(y_test, test_preds)
    test_r2 = r2_score(y_test, test_preds)

    print(f"\nTest Mean Squared Error (MSE): {test_mse:.4f}")
    print(f"Test R2 Coefficient of Determination: {test_r2:.4f}")

# ==========================================
# System Integrity Verification Module
# ==========================================
def test_pipeline_readiness():
    """
    System check framework: Verifies environmental dependencies and structural integrity
    of the ingestion pipeline before executing model allocation.
    """
    print("\n[SYSTEM CHECK] Initiating pre-execution environment verification...")

    # 1. Verify environmental token allocation
    token = os.environ.get("TABPFN_TOKEN", "")
    if not token or token.startswith("your_"):
        print("[ERROR] Verification Failed: No valid TABPFN_TOKEN detected in environment!")
        return False

    # 2. Structural column and integrity check
    data_dir = "mnt/data"
    target_files = ["train.csv", "valid.csv", "test.csv"]
    target_col = "<label>"

    for f in target_files:
        f_path = os.path.join(data_dir, f)
        # Check physical existence
        if not os.path.exists(f_path):
            print(f"[ERROR] Verification Failed: File path target missing -> {f_path}")
            return False

        # Structural testing via chunk ingestion to safeguard against corrupted/missing layouts
        try:
            df_check = pd.read_csv(f_path, nrows=5)
            if target_col not in df_check.columns:
                print(f"[ERROR] Structural Exception: Specified target '{target_col}' missing in '{f}'! Found trailing column: '{df_check.columns[-1]}'")
                return False
        except Exception as e:
            print(f"[ERROR] Ingestion Test Failure on dataset {f}: {e}")
            return False

    print("[SUCCESS] Pipeline verification and environmental integrity assertions passed. Proceeding to execution.")
    return True

if __name__ == "__main__":
    # TODO: Token injection must execute before running integrity checks
    os.environ["TABPFN_TOKEN"] = "<TABPFN TOKEN>"

    if test_pipeline_readiness():
        main()

# Baseline: FT-T and TabM

In [ ]:
!pip install pytorch_tabular #ft-t
!pip install tabm      #tabM
!pip install git+https://github.com/yandex-research/tabm.git  #tabM
!pip install rtdl_num_embeddings  #tabM
!pip install catboost   #catboost

dataset prepare

In [ ]:
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler

def prepare_multi_modal_data_3way(train_csv, valid_csv, test_csv, target_col='<label>'):
    """
    Read the Train, Valid, Test CSV files and do a normalization.
    """
    print("Loading...")
    train_df = pd.read_csv(train_csv)
    valid_df = pd.read_csv(valid_csv)
    test_df = pd.read_csv(test_csv)

    # check column consistency
    train_cols = list(train_df.columns)
    if list(valid_df.columns) != train_cols or list(test_df.columns) != train_cols:
        raise ValueError("The cols of train/valid/test are inconsistent...")

    # classify column
    id_cols = ['CaseNo']
    methy_cols = [c for c in train_cols if c.startswith('cg')]
    metadata_cols = [c for c in train_cols if c.startswith('ISLAND_')]

    # clinical column
    clinical_cols = [c for c in train_cols if c not in (methy_cols + metadata_cols + id_cols + [target_col])]

    print(f"   - Methylation: {len(methy_cols)} features")
    print(f"   - Probe Metadata: {len(metadata_cols)} features")
    print(f"   - Clinical Others: {len(clinical_cols)} features")

    all_baseline_features = methy_cols + metadata_cols + clinical_cols

    # Standardization
    scaler = StandardScaler()
    train_df[all_baseline_features] = scaler.fit_transform(train_df[all_baseline_features])

    valid_df[all_baseline_features] = scaler.transform(valid_df[all_baseline_features])
    test_df[all_baseline_features] = scaler.transform(test_df[all_baseline_features])

    print(f"   - Train shape: {train_df.shape}")
    print(f"   - Valid shape: {valid_df.shape}")
    print(f"   - Test shape:  {test_df.shape}")

    return train_df, valid_df, test_df, methy_cols, metadata_cols, clinical_cols

In [ ]:
# Data Preparation
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

train_path = "<path of training set>"
valid_path = "<path of valid set>"
test_path = "<path of test set>"
train_df, valid_df, test_df, methy_cols, metadata_cols, clinical_cols = prepare_multi_modal_data_3way(train_csv=train_path,
                                                 valid_csv=valid_path,
                                                 test_csv=test_path)
target_col = '<label>'

all_baseline_features = methy_cols + metadata_cols + clinical_cols

train_labels = train_df[target_col].values
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)

print(f"class weights [Class 0, Class 1]: {class_weights}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

catboost

In [ ]:
# ==========================================
# CatBoost Training & Early Stopping
# (Assumes Data Preparation cell has already been run)
# ==========================================
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
import numpy as np

# Extract X and y using the previously defined variables in your notebook
X_train, y_train = train_df[all_baseline_features], train_df[target_col]
X_valid, y_valid = valid_df[all_baseline_features], valid_df[target_col]
X_test, y_test   = test_df[all_baseline_features], test_df[target_col]

# Convert the existing class_weights array to a dictionary for CatBoost
classes = np.unique(y_train)
class_weights_dict = dict(zip(classes, class_weights))

print(f"\nStarting CatBoost training (Features: {len(all_baseline_features)})...")

# Initialize CatBoost model
catboost_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.03,
    depth=6,
    class_weights=class_weights_dict,
    eval_metric='AUC',
    random_seed=42,
    verbose=100
)

# Train with early stopping using the validation set
catboost_model.fit(
    X_train, y_train,
    eval_set=(X_valid, y_valid),
    early_stopping_rounds=50,
    use_best_model=True
)

print("Training complete! Best model weights automatically loaded.")


# ==========================================
# Testing and Evaluation
# ==========================================
print("\nPerforming comprehensive evaluation on the test set...")

# Predict probabilities (for AUC) and class labels (for other metrics)
probs = catboost_model.predict_proba(X_test)[:, 1]
preds = catboost_model.predict(X_test)
targets = y_test.values

# --- Calculate Metrics ---
acc = accuracy_score(targets, preds)
auc = roc_auc_score(targets, probs)

tn, fp, fn, tp = confusion_matrix(targets, preds).ravel()

recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

# --- Print highly detailed report ---
print("========================================")
print("【CatBoost Test Set Performance】")
print(f"Accuracy:              {acc:.4f}")
print(f"ROC AUC:               {auc:.4f}")
print(f"Recall / Sensitivity:  {recall:.4f} (Ability to detect positives)")
print(f"Specificity:           {specificity:.4f} (Ability to exclude negatives)")
print(f"Precision:             {precision:.4f} (Reliability of positive predictions)")
print(f"F1-Score:              {f1:.4f}")
print("========================================")

FT - Transformer

In [ ]:
#FT-T
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
import numpy as np
import torch
import torch.nn as nn

# ---------------------------------------------------------
# 3. Initialization and Training (with Class Weighting)
# ---------------------------------------------------------
tabular_model = TabularModel(
    data_config=data_config,
    model_config=model_config,
    optimizer_config=optimizer_config,
    trainer_config=trainer_config,
)

# Calculate class weights for the training set
train_labels = train_df[target_col].values
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)

# Convert to PyTorch Tensor
# Note: .to(device) is not required here as the underlying PyTorch Tabular Lightning
# implementation handles device conversion automatically.
weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

# Define weighted Loss Function
weighted_loss = nn.CrossEntropyLoss(weight=weights_tensor)

print("Starting FT-Transformer Baseline training with real data (Class Weights enabled)...")

# [Key Modification]: Pass the weights to the model via the loss parameter
tabular_model.fit(
    train=train_df,
    validation=valid_df,
    loss=weighted_loss
)

print("Training complete!")

# ---------------------------------------------------------
# 4. Evaluation and Testing (Extracting Essential Paper Metrics)
# ---------------------------------------------------------
print("\nPerforming comprehensive evaluation on the test set...")

# pytorch_tabular's predict returns a DataFrame containing prediction results
pred_df = tabular_model.predict(test_df)

# --- Dynamic Column Retrieval (Error Proofing) ---
# You can print all columns here if you're curious:
# print("Prediction DataFrame columns:", pred_df.columns.tolist())

targets = test_df[target_col].values

# 1. Retrieve the prediction probability column (look for columns containing 'prob' and '1')
prob_cols = [col for col in pred_df.columns if 'prob' in col.lower()]
prob_col_1 = [col for col in prob_cols if '1' in col]

if len(prob_col_1) > 0:
    probs = pred_df[prob_col_1[0]].values
else:
    # If no column containing '1' is found, usually the last or only 'prob' column
    # represents the positive class probability.
    probs = pred_df[prob_cols[-1]].values

# 2. Retrieve the prediction result (0 or 1) column (look for columns containing 'predict')
pred_col_candidates = [col for col in pred_df.columns if 'predict' in col.lower()]
if len(pred_col_candidates) > 0:
    preds = pred_df[pred_col_candidates[0]].values
else:
    # If still not found, derive predictions from the probabilities (probs)
    # Probability > 0.5 is predicted as positive (1), otherwise negative (0)
    print("Prediction label column not found; deriving from probabilities...")
    preds = (probs > 0.5).astype(int)

# --- Calculate Metrics ---
# Basic Metrics
acc = accuracy_score(targets, preds)
auc = roc_auc_score(targets, probs)

# Advanced Biomedical Metrics
tn, fp, fn, tp = confusion_matrix(targets, preds).ravel()

recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

print("========================================")
print("【FT-Transformer Test Set Performance】")
print(f"Accuracy:              {acc:.4f}")
print(f"ROC AUC:               {auc:.4f}")
print(f"Recall / Sensitivity:  {recall:.4f} (Ability to detect positives)")
print(f"Specificity:           {specificity:.4f} (Ability to exclude negatives)")
print(f"Precision:             {precision:.4f} (Reliability of positive predictions)")
print(f"F1-Score:              {f1:.4f}")
print("========================================")

TabM

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import tabm
from rtdl_num_embeddings import LinearReLUEmbeddings
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report
import copy

# ==========================================
# 1. build PyTorch Dataset
# ==========================================
class EWASDataset(Dataset):
    def __init__(self, df, feature_cols, target_col):
        self.x = torch.tensor(df[feature_cols].values, dtype=torch.float32)
        self.y = torch.tensor(df[target_col].values, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

# ==========================================
# 2. training / valid function
# ==========================================
def train_tabm_with_early_stopping(model, train_loader, valid_loader, optimizer, criterion, device, max_epochs=50, patience=5):
    print(f"Starting TabM training (Max Epochs: {max_epochs}, Patience: {patience})")

    best_valid_loss = float('inf')
    best_weights = copy.deepcopy(model.state_dict())
    patience_counter = 0

    for epoch in range(max_epochs):
        # --- Training Phase ---
        model.train()
        train_loss = 0.0

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()

            logits = model(x) # shape: [batch_size, k, num_classes]

            batch_loss = sum(criterion(logits[:, i, :], y) for i in range(model.k)) / model.k

            batch_loss.backward()
            optimizer.step()
            train_loss += batch_loss.item() * x.size(0)

        train_loss /= len(train_loader.dataset)

        # --- Validation Phase ---
        model.eval()
        valid_loss = 0.0

        with torch.no_grad():
            for x, y in valid_loader:
                x, y = x.to(device), y.to(device)
                logits = model(x)
                batch_loss = sum(criterion(logits[:, i, :], y) for i in range(model.k)) / model.k
                valid_loss += batch_loss.item() * x.size(0)

        valid_loss /= len(valid_loader.dataset)

        print(f"Epoch [{epoch+1:02d}/{max_epochs}] - Train Loss: {train_loss:.4f} | Valid Loss: {valid_loss:.4f}")

        # --- Early Stopping check ---
        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            best_weights = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early Stopping triggered! (Valid loss did not improve for {patience} consecutive epochs)")
                break

    print("Loading best model weights...")
    model.load_state_dict(best_weights)
    return model

# ==========================================
# 3. test / evaluate function
# ==========================================
def evaluate_tabm(model, test_loader, device):
    print("\nRunning comprehensive evaluation on the test set...")
    model.eval()

    all_preds = []
    all_probs = []
    all_targets = []

    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)

            test_logits = model(x)
            probs = F.softmax(test_logits, dim=-1)

            final_probs = probs.mean(dim=1)
            predictions = final_probs.argmax(dim=-1)

            all_preds.extend(predictions.cpu().numpy())
            all_probs.extend(final_probs[:, 1].cpu().numpy())
            all_targets.extend(y.cpu().numpy())

    # --- Metrics ---
    acc = accuracy_score(all_targets, all_preds)
    auc = roc_auc_score(all_targets, all_probs)

    # --- TN, FP, FN, TP ---
    # tn: True Negative, fp: False Positive
    # fn: False Negative, tp: True Positive
    tn, fp, fn, tp = confusion_matrix(all_targets, all_preds).ravel()

    # --- 3. bio Metrics ---
    # Recall (Sensitivity): TP / (TP + FN)
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

    # Specificity: TN / (TN + FP)
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    # Precision: TP / (TP + FP)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0

    # F1-Score
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

    # --- report ---
    print("========================================")
    print(f"Accuracy:    {acc:.4f}")
    print(f"ROC AUC:     {auc:.4f}")
    print(f"Recall / Sensitivity: {recall:.4f} (Ability to detect positives)")
    print(f"Specificity: {specificity:.4f} (Ability to exclude negatives)")
    print(f"Precision:   {precision:.4f} (Precision of positive predictions)")
    print(f"F1-Score:             {f1:.4f}")
    print("========================================")


    return acc, auc, recall, specificity, f1

if __name__ == "__main__":


    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    target_col = '<label>'
    all_features = methy_cols + metadata_cols + clinical_cols
    n_features = len(all_features)

    print(f"Total input feature dimension: {n_features} (Methylation + Metadata + Clinical)")

    batch_size = 32
    train_loader = DataLoader(EWASDataset(train_df, all_features, target_col), batch_size=batch_size, shuffle=True)
    valid_loader = DataLoader(EWASDataset(valid_df, all_features, target_col), batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(EWASDataset(test_df, all_features, target_col), batch_size=batch_size, shuffle=False)


    model = tabm.TabM.make(
        n_num_features=n_features,
        num_embeddings=LinearReLUEmbedembeddings(n_features),
        d_out=2,
        k=8
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-1)
    criterion = nn.CrossEntropyLoss(weight=weights_tensor)

    model = train_tabm_with_early_stopping(
        model=model,
        train_loader=train_loader,
        valid_loader=valid_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        max_epochs=50,
        patience=5
    )
    evaluate_tabm(model, test_loader, device)

xgboost

In [ ]:
!python <path of baseline model root>/xgboost_ewas_baseline.py \
  --train_csv <path of trainingset> \
  --valid_csv <path of validset> \
  --test_csv <path of testset> \
  --output_dir <outputpath> \
  --cpg_start_excel <cpg start column> \
  --cpg_end_excel <cpg end column> \
  --scale_pos_weight sqrt \
  --n_estimators 200 \
  --max_depth 2 \
  --learning_rate 0.001 \
  --subsample 0.8 \
  --colsample_bytree 0.8 \
  --min_child_weight 1 \
  --reg_lambda 5.0 \
  --reg_alpha 0.0 \
  --seed 0

pure Tansformer

In [ ]:
!python <path of baseline root>/pure_transformer_ewas_baseline.py \
  --train_csv <path of trainingset>\
  --valid_csv <path of validset> \
  --test_csv <path of testset> \
  --output_dir <outputpath> \
  --cpg_start_excel <cpg start column> \
  --cpg_end_excel <cpg end column> \
  --embed_dim 128 \
  --num_layers 2 \
  --heads 1 \
  --dropout 0.25 \
  --pooling cls \
  --batch_size 16 \
  --epochs 50 \
  --lr 5e-4 \
  --weight_decay 1e-4 \
  --scheduler cosine \
  --monitor_metric f1 \
  --loss_type combo \
  --loss_weight_scale 0.5 \
  --device cuda